In [74]:
import pandas as pd
from datasets import load_dataset
from functions.pred import *
from functions.xai import *
from functions.eval import *
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import os

In [75]:
ds = load_dataset("manueltonneau/hateday", token=os.environ.get("HF_TOKEN_PATH"))
df = pd.DataFrame(ds["train"])

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: afaad7c1-2550-4329-b77a-1222926fc9c2)')' thrown while requesting HEAD https://huggingface.co/datasets/manueltonneau/hateday/resolve/main/README.md
Retrying in 1s [Retry 1/5].


In [76]:
df.lang_country_hateday.value_counts()

lang_country_hateday
US         45000
ar         45000
de         45000
en         45000
es         45000
fr         45000
in         45000
india      45000
kenya      45000
nigeria    45000
pt         45000
tr         45000
Name: count, dtype: int64

In [77]:
df = df[df.lang_country_hateday == "ar"]
df.shape

(45000, 10)

In [78]:
df.columns

Index(['tweet_id', 'text', 'class_clean', 'twitter_hate', 'violent_hate',
       'target_majority', 'target_category', 'total_engagement', 'weighted',
       'lang_country_hateday'],
      dtype='str')

In [79]:
df.class_clean.value_counts()

class_clean
0    43842
1      996
2      162
Name: count, dtype: int64

In [80]:
df["class_clean"] = df["class_clean"].map({0: "Neutral", 1: "Hate", 2: "Hate"})

In [81]:
# removing @user, RT, http from the text
df["text"] = df["text"].str.replace("@USER", "")
df["text"] = df["text"].str.replace("LINK", "")
df["text"] = df["text"].str.replace("http", "")

In [82]:
# remove multiple spaces
df["text"] = df["text"].str.replace("\s+", " ", regex=True)
df["text"] = df["text"].str.strip()

<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_23200\3601961224.py:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df["text"] = df["text"].str.replace("\s+", " ", regex=True)


In [83]:
df["text"] = df["text"].apply(remove_chaklas)

In [84]:
df.class_clean.value_counts()

class_clean
Neutral    43842
Hate        1158
Name: count, dtype: int64

In [85]:
df.rename(columns={"class_clean": "label"}, inplace=True)

In [86]:
df = df.groupby("label", group_keys=False).sample(n=1000, random_state=0).reset_index(drop=True)

In [87]:
df.label.value_counts()

label
Hate       1000
Neutral    1000
Name: count, dtype: int64

In [88]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "IbrahimAmin/marbertv2-finetuned-egyptian-hate-speech-detection"
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device)

Device set to use cuda


In [89]:
df["MarBERT"] = df["text"].apply(lambda x: predict_class(pipe, x))

In [90]:
accuracy = accuracy_score(df["label"], df["MarBERT"])
precision = precision_score(df["label"], df["MarBERT"], average="weighted")
recall = recall_score(df["label"], df["MarBERT"], average="weighted")
f1 = f1_score(df["label"], df["MarBERT"], average="weighted")
print("MarBERT")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

MarBERT
Accuracy: 0.841
Precision: 0.8519004689254344
Recall: 0.841
F1-score: 0.8397590944272447


In [91]:
df = df[["text", "label", "MarBERT"]]
df

,text,label,MarBERT
0,اللهم انتقم منو أشد انتقام #عباس_الزفر_خرب_الز...,Hate,Hate
1,ياعمي هذا ترعرع في الفكر البعثي تجده الان في ح...,Hate,Hate
2,اقسم بالله الأشكال ذي ما تفيد معهم المسامحة وا...,Hate,Hate
3,نجلخ ؟! 🔥💦 #فويس - #لاين #معصيتي_راحتي #مشتهيه...,Hate,Hate
4,اي حد وانت مغمض يقولك لازم مجلس الخطيب يستقيل ...,Hate,Hate
...,...,...,...
1995,مايووحشك غالي لعد تعيدينها تويتر كان خايس,Neutral,Neutral
1996,جان من #ایران_من,Neutral,Hate
1997,الاشوري قومية وعرق الاثوري مذهب من مذاهب المسي...,Neutral,Hate
1998,أشهر فروع أشراف الحجاز والمخلاف السليماني واليمن,Neutral,Hate


In [92]:
df.to_csv("data/model_pred_hate/model_pred_hate.csv", index=False)